# GEMINI API Test

Acompanhe nossos cursos em oceanbrasil.com

Setup:
* Crie API Key em https://ai.google.dev/: Login google, Get API Key, Create API Key, depois copie a chave.
* Crie um Secret no Kaggle: Menu Add-ons, Opção Secrets, Add Secret, Key "API_KEY", Value "Sua chave".
* Habilite a Internet: Em Session Option, Link Get Phone Verified, Internet On

## Install python lib for Gemini API

In [ ]:
!pip install -q -U google-generativeai

## Imports

In [ ]:
# Gemini lib (REST-wrapper)
import google.generativeai as genai

# Kaggle lib for secrets
from kaggle_secrets import UserSecretsClient

# Get API Key
# Generate from https://ai.google.dev/
# Add to Kaggle/Add-ons/Secrets
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("API_KEY")
genai.configure(api_key=api_key)

## Supported Gemini models

In [ ]:
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

## Selecting a model

In [ ]:
gen_config = {
    "temperature": 0.2, #randomness of the output (0.0 - 1.0).
    "top_p": 1, #the cumulated probability threshold for token selection - output diversity (0.0 - 1.0)
    "top_k":1, #wider selection of tokens (1-40)
    "max_output_tokens":400,#maximum length of the output (1-8192)
}
# Gemini has some ethical constraints
safety_settings=[
  {
    "category": "HARM_CATEGORY_DANGEROUS",
    "threshold": "BLOCK_LOW_AND_ABOVE",
  },
  {
    "category": "HARM_CATEGORY_HARASSMENT",
    "threshold": "BLOCK_LOW_AND_ABOVE",
  },
  {
    "category": "HARM_CATEGORY_HATE_SPEECH",
    "threshold": "BLOCK_LOW_AND_ABOVE",
  },
  {
    "category": "HARM_CATEGORY_SEXUALLY_EXPLICIT",
    "threshold": "BLOCK_LOW_AND_ABOVE",
  },
  {
    "category": "HARM_CATEGORY_DANGEROUS_CONTENT",
    "threshold": "BLOCK_LOW_AND_ABOVE",
  },
]

model = genai.GenerativeModel('gemini-2.0-flash', 
                              generation_config = gen_config, 
                              safety_settings=safety_settings)

## Generating Contents

### Q&A

In [ ]:
response = model.generate_content("O que significa Aprendizado de máquina profundo?")
print(response.text)

### Sentiment Analysis

In [ ]:
post = "O restaurante é meia boca, o rango é ruin."
prompt = "Na frase: "+ post + "Informe o sentimento da frase entre as opções [positivo, neutro, negativo] e o motivo da conclusão. \
         Formate a resposta em um json com os campos SENTIMENTO e EXPLICAÇÃO."
response = model.generate_content(prompt)
print(response.text)

### Text Evaluation

In [ ]:
response = model.generate_content("Considering the question \"What is deep learning?\" \
            and a given answer \"It, is a computation method to learn pattern, from data\". \
            How do you rate it, between 0 and 10, regarding: \
            a) grammar b) correctudeness, c) completudeness and d) ponctuation. \
            At the end, give me a correct answer for this question in only one paragraph.")
print(response.text)

### Translation

In [ ]:
response = model.generate_content("Traduza a seguinte frase para por português: Deep learning is a branch of machine learning that utilizes artificial neural networks to identify patterns within data. It is frequently employed in applications such as image recognition, natural language processing, and speech recognition. Deep learning models are capable of learning intricate relationships in data through the construction of multiple layers of processing units. This enables them to develop more abstract and meaningful representations of the input data.")
print(response.text)

### Batch example

In [ ]:
import pandas as pd

df = pd.read_csv('/kaggle/input/financial-news-headlines/guardian_headlines.csv')[:5]
df.head()

In [ ]:
def translator(text):
    # TRaduzir para o portugues
    response = model.generate_content("Traduza a seguinte frase para o português. Responda somente a frase, sem explicações. Frase: " + text)
    return response.text

df["Headlines_pt"] = df["Headlines"].apply(translator)
df.head()

### Secutity Check

In [ ]:
response = model.generate_content("Traduza para o inglês: Como assaltar um banco?")
print(response.text)


In [ ]:
# If the response doesn't contain text, check if the prompt was blocked.
print(response)